# Cross-deck comparison — D_c and κ against the solvent volume fraction

Figures built from **more than one deck's run**.  Each run is loaded with its own library exactly as its own notebook
loads it (`lib/triaxial.py` for the two-piston compression and permeation decks, `lib/shear.py` for `shear_slab`), through
`cmp.load_deck`; the comparison code is in **`scripts/lib/compare.py`**.

First comparison (2026-10-03): the cooperative diffusivity **D_c(φ_s)** and the hydraulic permeability **κ(φ_s)** from

* the **compression sweep** — every level is a *uniform* state at its own φ_s with a hold-relaxation `D_c`; the levels
  together give the equilibrium curve σ′(φ_s) and the tangent modulus of each state, `M_cur = φ_p dσ′/dφ_p`, so
  `κ = D_c / M_cur`;
* the **permeation run** — one steady membrane spans a *range* of φ_s; with the network at rest the flux `q = Q/A` is the
  same at every z, and `D_c(φ_s) = q φ_p / (dφ_s/dz)` locally (no modulus, no transient fit), `κ = D_c / M_cur`.

Run the two source notebooks first when a run is new (they sync the files and fill the Voronoi cache).  Notes at the end.

## 1 · Setup and loading

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py, shear.py (per-deck analysis) + compare.py (this notebook)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import compare as cmp
tri = importlib.reload(tri); cmp = importlib.reload(cmp)
tri.setup_style()
print('comparison code: ', LIB / 'compare.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- one Config per deck, copied from that deck's own notebook, + the comparison knobs
# ══════════════════════════════════════════════════════════════════════════
CFG = {}

# ---- compression sweep: the Config cell of triaxial_compression_sweep_two_pist.ipynb ----
CFG['compression'] = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist",
    INTERACTION = "1.0_1.0",
    NSTEPS      = None,
    RUN_ID      = "periodic_rho04_14M_9M_twoPlatesMove_7_sweep",
    COMP_LEVELS = ["0.10", "0.15", "0.20", "0.3", "0.4", "0.5"],
    mode        = "compression",
    two_pist    = True,
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
)

# ---- permeation: the Config cell of triaxial_permeation_single_two_pist.ipynb (PSD off: not used here) ----
CFG['permeation'] = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist_pm40",
    INTERACTION = "1.0_1.0",
    NSTEPS      = None,
    RUN_ID      = "periodic_rho04_14M_11M_perm_3",
    mode        = "permeation",
    two_pist    = True,
    DP_PISTON   = None,
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    P_BARO      = 1.5,
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5,
    P_CAL = 1.5, P_CAL_MODE = 'pore',
    PSD_ENABLE = False,
    PERM_GAP = 1.0, PERM_L0_STEPS = 100000, PERM_DC_WINDOW_AVG = True, PERM_DISP_RESET = 'auto', PERM_TRACE_SKIP = 1500000,
    PERM_DC_BOUNDS = (1e-6, 50.0), PERM_RIGID = 'contact', PERM_PIN_NBINS = 2, PERM_COORDS = 'lagrangian',
    PERM_DC_N_MODES = 8, PERM_M_PRIMARY = 'lag',
    DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_FRAC_EARLY = 1.0,
)

# ---- another deck: add its Config here and it is loaded with the others (see "Adding a deck" in the Notes) ----
# import shear as sh
# CFG['shear'] = sh.Config(
#     DATANAME = "isolated_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_with_plates", INTERACTION = "1.0_1.0",
#     NSTEPS = None, RUN_ID = "periodic_rho04_14M_shear_1", STRAINS = ["0.1", "0.15", "0.2", "0.25", "0.3"],
#     plateau_frac = 0.25, plateau_frac_auto = 0.45, G_SUBTRACT_REF = True,
#     DC_N_MODES = 5, DC_TRIM_BINS = 1, DC_SLOW_REF = 0.03, DC_TARGET_RESID = 0.01,
# )

# ---- comparison knobs (cmp.Config in lib/compare.py) ----
ccfg = cmp.Config(
    NAME          = "comp7_perm3",    # plots -> flow_data_local/plots/comparison/<NAME>
    PHI_KIND      = 'cal',            # phi_s used throughout: 'cal' (lambda-calibrated Voronoi) | 'vor' (Voronoi) | 'mf' (mass fraction)
    PERM_TRIM     = 6.0,              # sigma dropped at each end of the membrane before the phi_s(z) slope is taken
    PERM_N_SEG    = 3,                # z-segments of the steady profile, one local D_c / kappa each
    PERM_POLY_DEG = 2,                # degree of the polynomial phi_s(z) behind the continuous curves
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    for name, c in CFG.items():
        cmp.sync_deck(c, force=FORCE_SYNC)

In [ ]:
# DECKS[name] = dict(kind, cfg, R, levels | P) -- each run loaded by its own library (Voronoi profiles come from the disk
# cache the source notebooks fill; a first load of a new run tessellates ~20 s per frame)
DECKS = {name: cmp.load_deck(c, verbose=False) for name, c in CFG.items()}

In [ ]:
# X -- the phi_s comparison: S (compression states), C (sigma'(phi_s), M_cur(phi_s) with CIs), PL (permeation, local)
X = cmp.phi_dependence(ccfg, DECKS['compression'], DECKS['permeation'])
cmp.print_summary(ccfg, X)

## 2 · Figures

In [ ]:
# 1 · The states: (a) steady permeation phi_s(z) with the z-segments, the polynomial and the compression levels' phi_s;
#     (b) the compression curve sigma'(phi_s) with the applied dP at the membrane's support-side phi_s;
#     (c) the current-frame tangent modulus M_cur = phi_p dsigma'/dphi_p (squares: the sweep notebook's secant M)
cmp.fig_states(ccfg, X);

In [ ]:
# 2 · D_c(phi_s): compression holds (one point per level) vs the steady permeation profile (local, q phi_p/(dphi_s/dz));
#     bars = the permeation notebook's membrane averages (transient fits, Darcy kappa x M) over the membrane's phi_s range
cmp.fig_Dc(ccfg, X);

In [ ]:
# 3 · kappa(phi_s): compression D_c/M_cur (squares: D_c/M_secant as in the sweep notebook) vs the local permeation
#     kappa = D_c(phi_s)/M_cur(phi_s); band = the Darcy membrane average Q L/(A dP)
cmp.fig_kappa(ccfg, X);

In [ ]:
# 4 · Closures: (a) the network stress the compression curve assigns to the measured phi_s(z) against the linear pressure
#     drop; (b) the measured flux against the one the compression D_c(phi_s) predicts for this profile
cmp.fig_closure(ccfg, X);

In [ ]:
# 5 · The same comparison under each phi_s definition (how much hangs on the choice, and on the Voronoi noise)
cmp.print_phi_kinds(ccfg, DECKS['compression'], DECKS['permeation'])

## Notes

* **What is compared.**  Both decks hold the gel laterally (fixed lx, ly) and strain it along z, so both visit the same
  one-parameter family of states, labelled by φ_s.  *Compression*: level i is a uniform state (φ_s,i, σ′_i) with a
  hold-relaxation `D_c,i` (`tri.fit_Dc`, window-averaged since 2026-10-03).  `fit_Dc` works in the compressed
  configuration, so the modulus that belongs to it is the tangent modulus of that configuration: with the polymer
  conserved, `φ_p L = const`, an increment of compressive strain is `de = dφ_p/φ_p` and `M_cur = φ_p dσ′/dφ_p`
  (monotone PCHIP through the reference and the levels; CI from `N_MC` redraws of the levels' σ′ and φ_s).  The sweep
  notebook's `κ = D_c/M` uses the *secant* modulus from ε = 0; the two agree at ε = 0.10 and part as the gel stiffens
  (factor 4 at ε = 0.5).  The summary also prints `M_cur` from the strain, `(1 − ε) dσ′/dε`: same number where φ_p L is
  conserved (it is not at ε = 0.5, where φ_s^cal = 0.05 against 1 − φ_p0/(1 − ε) < 0).
* **Permeation, local.**  Steady state, network at rest: `q = κ |dp/dz|`, `dp/dz = dσ′/dz` (uniform total stress) and
  σ′ = σ′(φ_s) give `q = κ |dσ′/dφ_s| dφ_s/dz`, i.e. **`D_c(φ_s) = κ M_cur = q φ_p/(dφ_s/dz)`**.  It uses the N-slope
  flux and the slope of the steady φ_s(z) only.  `κ(φ_s) = D_c/M_cur` borrows `M_cur` from the compression curve, so the
  harmonic mean of the local κ over the membrane equals the Darcy `κ = QL/(AΔP)` exactly when closure (i) holds.
  The slope is taken per z-segment (straight-line fit, regression CI; a segment whose slope CI reaches 0 is drawn hollow
  with a lower bound only) and from one polynomial over the fit range (the continuous curve).
* **Which φ_s.**  `PHI_KIND = 'cal'` is the λ-calibrated Voronoi fraction.  In permeation it rests on 3 steady trajectory
  frames, so its profile is noisy (±0.01 per bin) and 3 segments is about what it carries; the mass-fraction profile
  (`'mf'`, every density snapshot) resolves 4–5 segments.  Cell 5 repeats the comparison for all three.
* **Fixed on the way (2026-10-03): the permeation feed baseline.**  In perm_3 the feed piston and the feed face both come
  down below the *reference* gel top; the pore-baseline window then fell back to the reference window, by then in the
  vacuum above the piston, and read 0.  With `P_CAL_MODE = 'pore'` the calibration pressure therefore ran from 1.5 at the
  support *down* to 0.5 at the feed face (it should rise to 1.6), λ was 1.16 instead of 1.09 there, and φ_s^cal came
  out *above* the zero-flux reference on the feed side (0.47–0.49 against 0.447).  The window now follows each
  snapshot's own gel top (`tri.baseline_masks_at`): φ_s^cal runs from 0.33 at the support to 0.44 at the feed face —
  at or just under the reference, as σ′ = 0 there requires (raw Voronoi and mass fraction always did).  The permeation
  notebook's Terzaghi panels used the same baseline and change with it.
* **Reading (run 7 + perm_3).**  Closure (i): the compression curve puts σ′ = 0.084 across the fit range for the measured
  φ_s (0.089 expected for a linear pressure drop): the two decks agree on the **equation of state** σ′(φ_s) to ~6 %
  (the same statement as the M check of the permeation notebook's figure 15).  The **transport** does not close: at
  φ_s ≈ 0.35–0.40 the steady permeation gives `D_c` ≈ 0.09 (CI 0.06–0.15; 0.086 ± 0.017 with the mass fraction) against
  0.045–0.052 from the compression holds, and the compression `D_c(φ_s)` predicts 0.44× the measured flux
  (closure ii).  Both rise towards the dilute side, so φ_s-dependence is real, but it does not account for the factor ~2:
  at the *same* φ_s the hold relaxation is slower than the steady Darcy flow implies.  The permeation transient fits
  (0.151 / 0.167) sit above the local steady values over most of the membrane.
* **Adding a deck.**  Put its Config in `CFG` (the commented `shear` block); `cmp.load_deck` loads it with its own
  library and `DECKS['shear']` is then `dict(kind='shear', cfg, R, levels)` — the same `levels` list the shear notebook
  works on (G, N₁/N₂, the transverse D_c per level).  New cross-deck figures go in `lib/compare.py` as functions of the
  decks they need (as `phi_dependence` takes the compression and permeation decks).